# A short introduction to containerized software

After spending time using nf-core pipelines to answer bioinformatic questions, we will focus on the processes that lie behind these pipelines now.

Today, we will focus on containerization, namely via Docker. 



1. Check if Docker is installed.

In [1]:
!docker info

Client: Docker Engine - Community
 Version:    29.8.1
 Context:    default
 Debug Mode: false
 Plugins:
  buildx: Docker Buildx (Docker Inc.)
    Version:  v0.37.1
    Path:     /usr/libexec/docker/cli-plugins/docker-buildx
  compose: Docker Compose (Docker Inc.)
    Version:  v5.5.1
    Path:     /usr/libexec/docker/cli-plugins/docker-compose

Server:
 Containers: 3
  Running: 0
  Paused: 0
  Stopped: 3
 Images: 24
 Server Version: 29.8.1
 Storage Driver: overlayfs
  driver-type: io.containerd.snapshotter.v1
 Logging Driver: json-file
 Cgroup Driver: systemd
 Cgroup Version: 2
 Plugins:
  Volume: local
  Network: bridge host ipvlan macvlan null overlay
  Log: awslogs fluentd gcplogs gelf journald json-file local splunk syslog
 CDI spec directories:
  /etc/cdi
  /var/run/cdi
 Swarm: inactive
 Runtimes: io.containerd.runc.v2 runc
 Default Runtime: runc
 Init Binary: docker-init
 containerd version: ee2735368117d2eb259779949d5e75cdafec9761
 runc version: v1.5.1-0-g8f2685a4
 init version:

### What is a container?

*A container creates a sandbox environment, that is able to run standalone, executable packages of software. It contains a wrapper around everything. The container itself contains everything that is needed to run anapplication, such as code, runtime, system tools, system libraries, and settings. Also ensures reproducability, easy to run, isolated environment, solves conflicted tendencies if different version necessary.* 

### Why do we use containers?

*We use containers to ensure replicability and to keep everything in order. If we try to run differential gene expression for example on different OS systems we might get different numbers of expressed genes, this might be due to differences in dependencies. But if a docker container is used, this container should produce similar results for every user.* 

### What is a docker image?

*A docker image is standarized and includes all files, binaries, libraries, and configurations to run in a container.* 

### Let's run our first docker image:

### Login to docker

In [ ]:
# This you need to do on the command line directly
!docker login

### Run your first docker container

In [8]:
!docker run hello-world


Hello from Docker!
This message shows that your installation appears to be working correctly.

To generate this message, Docker took the following steps:
 1. The Docker client contacted the Docker daemon.
 2. The Docker daemon pulled the "hello-world" image from the Docker Hub.
    (amd64)
 3. The Docker daemon created a new container from that image which runs the
    executable that produces the output you are currently reading.
 4. The Docker daemon streamed that output to the Docker client, which sent it
    to your terminal.

To try something more ambitious, you can run an Ubuntu container with:
 $ docker run -it ubuntu bash

Share images, automate workflows, and more with a free Docker ID:
 https://hub.docker.com/

For more examples and ideas, visit:
 https://docs.docker.com/get-started/



### Find the container ID

In [ ]:
!docker ps --all

CONTAINER ID   IMAGE                                                         COMMAND                  CREATED         STATUS                      PORTS     NAMES
cb66915af451   hello-world                                                   "/hello"                 4 seconds ago   Exited (0) 3 seconds ago              serene_albattani
6b307ef574e4   community.wave.seqera.io/library/gzip_wget:041a8ba200c3fa3f   "/bin/bash -c 'eval …"   45 hours ago    Exited (137) 45 hours ago             nxf-QPORdnzu1EQdfmU3uhIOgJpF
2122941886cc   community.wave.seqera.io/library/gzip_wget:041a8ba200c3fa3f   "/bin/bash -c 'eval …"   46 hours ago    Exited (137) 45 hours ago             nxf-mgAhiY35tym22Kq3oNytyg1h
e4f3bc296001   hello-world                                                   "/hello"                 3 days ago      Exited (0) 3 days ago                 ecstatic_hypatia


!docker run --rm => removes container after running

### Delete the container again, give prove its deleted

In [10]:
!docker rm cb66915af451

!docker ps -a

cb66915af451
CONTAINER ID   IMAGE                                                         COMMAND                  CREATED        STATUS                      PORTS     NAMES
6b307ef574e4   community.wave.seqera.io/library/gzip_wget:041a8ba200c3fa3f   "/bin/bash -c 'eval …"   45 hours ago   Exited (137) 45 hours ago             nxf-QPORdnzu1EQdfmU3uhIOgJpF
2122941886cc   community.wave.seqera.io/library/gzip_wget:041a8ba200c3fa3f   "/bin/bash -c 'eval …"   46 hours ago   Exited (137) 45 hours ago             nxf-mgAhiY35tym22Kq3oNytyg1h
e4f3bc296001   hello-world                                                   "/hello"                 3 days ago     Exited (0) 3 days ago                 ecstatic_hypatia


### FASTQC is a very useful tool as you've learned last week. Let's try and run it from command line

Link to the software: https://www.bioinformatics.babraham.ac.uk/projects/fastqc/

Please describe the steps you took to download and run the software for the example fastq file from last week below:

*1. downloaded fastqc in miniconda with command: conda install fastqc (Its part of bioconda, if bioconda channel is not already ste up, it would also need to be set up)*

*2. run fastqc with command in terminal (and activated conda environment): fastqc /home/nina/computational-workflows-2026/day_03/example_fastq/SRX19144486_SRR23195516_1.fastq.gz*


### Very well, now let's try to make use of its docker container

1. create a container holding fastqc using seqera containers (https://seqera.io/containers/)
2. use the container to generate a fastqc html of the example fastq file

In [1]:
# pull the container
!docker pull community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae

0.12.1--9971ea336a9eddae: Pulling from library/fastqc
Digest: sha256:f97f7658a191bebe2f2a007cab5b815d380f89a948d6eb1481b046ab7618bebc
Status: Image is up to date for community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae
community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae


In [11]:
# run the container and save the results to a new "fastqc_results" directory
!mkdir -p /home/nina/computational-workflows-2026/day_04/fastqc_results

!docker run --rm \
    --user 1000:1000 \
  -v /home/nina/computational-workflows-2026/day_04:/fastqc_test \
 community.wave.seqera.io/library/fastqc:0.12.1--9971ea336a9eddae \
  fastqc /fastqc_test/SRX19144486_SRR23195516_1.fastq.gz -o /fastqc_test/fastqc_results /fastqc_test/SRX19144486_SRR23195516_1.fastq.gz

application/gzip
application/gzip
Started analysis of SRX19144486_SRR23195516_1.fastq.gz
Approx 5% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 10% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 15% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 20% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 25% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 30% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 35% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 40% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 45% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 50% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 55% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 60% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 65% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 70% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 75% complete for SRX19144486_SRR23195516_1.fastq.gz
Approx 80% complete for SRX

### Now that you know how to use a docker container, which approach was easier and which approach will be easier in the future?

*The miniconda approach was easier. Since I have already done that a lot. Both approaches are rather similar effort wise (in my optinion at least). So in the future if I am just doing an analysis for exercises I would probably still use miniconda. Though considering the advantages, I'd probably try to use docker if I was working on something that should be reproducible.* 

### What would you say, which approach is more reproducible?

*I would say the docker approach is more reproducible. As for the miniconda environment, i would need to specifiy every single detail for the steps I took and what version I installed. While for the docker approach I could just create a docker image, and than I would just have to share my docker image.*

### Compare the file to last weeks fastqc results, are they identical?
### Is the fastqc version identical?

*For last Week I had the multiqc report from the star_salmon run and fastqc files from running just Salmon. I decided to compare the miniconda and the docker approach with the running of just salmon fastqc. <br>
For day 2 there were two fastqc report, one with trimmed data and one with the raw data. I further decided to use the raw data fastqc report, as there was no trimming for both the docker and the miniconda approach. <br>
The basic parameters, GC count, sequence lenght, total bases, etc. were the same for all three files. <br>
The per base sequence quality, also appeared to be the same. <br>
The quality control graphs and tables also were the same. Though some scaling was different for the miniconda approach.*




## Dockerfiles

We now used Docker containers and images directly to boost our research. 

Let's create our own toy Dockerfile including the "cowsay" tool (https://en.wikipedia.org/wiki/Cowsay)

Hints:
1. Docker is Linux, so you need to know the apt-get command to install "cowsay"

In [ ]:
# open the file "my_dockerfile" in a text editor
!RUN apt-get update && apt-get install -y curl cowsay 

!ENV PATH="/usr/games:$PATH"

### Explain the RUN and ENV lines you added to the file

*The RUN command tells the builder to run a specified command.* <br>
*!RUN apt-get update && apt-get install -y curl cowsay* <br>
*Here the RUN command commands the builder to* <br>
*1. update the package lists for upgrades for packages that might need upgrading* <br>
*2. apt-get install -y curl cowsay, this installs the command line tool curl, and cowsay over apt-get.* <br> <br>
*ENV PATH="/usr/games:$PATH" command sets an environment variable that a running container will use, in this case it defines the Path where the programm looks for exectuables.*


In [1]:
# build the docker image
!docker build --pull --no-cache -f my_dockerfile -t my-cowsay .


[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.2s (1/2)                                          docker:default
 => [internal] load build definition from my_dockerfile                    0.0s
 => => transferring dockerfile: 806B                                       0.0s
 => [internal] load metadata for docker.io/library/debian:bookworm-slim    0.2s
[+] Building 0.3s (1/2)                                          docker:default
 => [internal] load build definition from my_dockerfile                    0.0s
 => => transferring dockerfile: 806B                                       0.0s
 => [internal] load metadata for docker.io/library/debian:bookworm-slim    0.3s
[+] Building 0.5s (1/3)                                          docker:default
 => [internal] load build definition from my_dockerfile                    0.0s
 => => transferring dockerfile: 806B                                       0.0s
 => [internal] load metadata for docker

In [2]:
# make sure that the image has been built
!docker images

                                                            i Info →   U  In Use
IMAGE                           ID             DISK USAGE   CONTENT SIZE   EXTRA
community.wave.seqera.io/library/atlas-gene-annotation-manipulation:1.1.1--b9e8c32c92709512
                                ecd8ae9a41c0       2.52GB          620MB        
community.wave.seqera.io/library/bioconductor-deseq2_bioconductor-limma:b56a0c9ddc3e87e1
                                e220c341c297       2.56GB          649MB        
community.wave.seqera.io/library/bioconductor-tximeta_jq:78bccd386c46a07c
                                b8b4a40e7c12       3.27GB          817MB        
community.wave.seqera.io/library/coreutils_grep_gzip_lbzip2_pruned:838ba80435a629f8
                                63c2c6b22e83        237MB         58.7MB        
community.wave.seqera.io/library/csvtk:0.37.0--113625988dd3285d
                                b13e22e5ecee        206MB           52MB        
community.wave.seqera.io/libra

In [4]:
# run the docker file 
!docker run --rm my-cowsay cowsay "Hello"

 _______
< Hello >
 -------
        \   ^__^
         \  (oo)\_______
            (__)\       )\/\
                ||----w |
                ||     ||


## Let's do some bioinformatics with the docker file and create a new docker file that holds the salmon tool used in rnaseq

To do so, use "curl" in your new dockerfile to get salmon from https://github.com/COMBINE-lab/salmon/releases/download/v1.5.2/salmon-1.5.2_linux_x86_64.tar.gz

In [ ]:
# use the file "salmon_docker" in this directory to build a new docker image


In [5]:
# build the image
!docker build --pull --no-cache -f salmon_docker -t salmon_exercise .

[+] Building 0.0s (0/1)                                          docker:default
[+] Building 0.2s (2/3)                                          docker:default
 => [internal] load build definition from salmon_docker                    0.0s
 => => transferring dockerfile: 561B                                       0.0s
 => [internal] load metadata for docker.io/library/debian:bookworm-slim    0.2s
 => [auth] library/debian:pull token for registry-1.docker.io              0.0s
[+] Building 0.4s (2/3)                                          docker:default
 => [internal] load build definition from salmon_docker                    0.0s
 => => transferring dockerfile: 561B                                       0.0s
 => [internal] load metadata for docker.io/library/debian:bookworm-slim    0.3s
 => [auth] library/debian:pull token for registry-1.docker.io              0.0s
[+] Building 0.5s (2/3)                                          docker:default
 => [internal] load build definition fro

In [6]:
# run the docker image to give out the version of salmon
!docker run --rm salmon_exercise salmon --version


salmon 1.5.2


## 1. Do you think bioinformaticians have to create a docker image every time they want to run a tool?

2. Find the salmon docker image online and run it on your computer.

3. What is https://biocontainers.pro/?

1. *No, There are many already ready to go Docker container, similar to packages in python, that can be used or build upon.*

3. *Biocontainers is a project that maintains containers and tools, mainly but not only for bioinformatics and biology.*

In [7]:
# 2.
!docker pull quay.io/biocontainers/salmon:2.8.0--hfa8f182_0 

2.8.0--hfa8f182_0: Pulling from biocontainers/salmon
Digest: sha256:deee6c1353277c3fa9b579a79eff18f1588ad0a9f098c299d583c49d3613be9b
Status: Image is up to date for quay.io/biocontainers/salmon:2.8.0--hfa8f182_0
quay.io/biocontainers/salmon:2.8.0--hfa8f182_0


In [8]:
!docker tag quay.io/biocontainers/salmon:2.8.0--hfa8f182_0 salmon_biocontainers:2.8.0
!docker run --rm salmon_biocontainers:2.8.0 salmon --version

salmon 2.8.0


## Are there other ways to create Docker (or Apptainer) images?

What is https://seqera.io/containers/ ?

*Yes there are other ways to create Docker (or Appcontainer) images. For example through the Sqera Containers API. Container Images are build using the Seqera Wave service. The requestes image is pushed to the Wave community registry for any subsequent request.* 